## RAG Pipelines- Data Ingestion to Vector DB Pipeline

In [3]:
import os
from langchain_community.document_loaders import (
    PyPDFLoader,
    PyMuPDFLoader
    )
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [4]:
# read all the pdf's inside the directory

def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir= Path(pdf_directory)

    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source information to metadata

            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")


Found 4 PDF files to process

Processing 25Fall_CS584_NLP_Project_Proposal_Neelam_Viyasha (2).pdf


Ignoring wrong pointing object 10 0 (offset 0)
Ignoring wrong pointing object 12 0 (offset 0)
Ignoring wrong pointing object 14 0 (offset 0)
Ignoring wrong pointing object 20 0 (offset 0)
Ignoring wrong pointing object 22 0 (offset 0)
Ignoring wrong pointing object 24 0 (offset 0)
Ignoring wrong pointing object 26 0 (offset 0)
Ignoring wrong pointing object 28 0 (offset 0)
Ignoring wrong pointing object 30 0 (offset 0)
Ignoring wrong pointing object 32 0 (offset 0)
Ignoring wrong pointing object 35 0 (offset 0)


  ✓ Loaded 3 pages

Processing NEELAM MORE_AI Builder Intern.pdf
  ✓ Loaded 1 pages

Processing NEELAM MORE_AI.pdf
  ✓ Loaded 1 pages

Processing nmore2@stevens.edu_HW1.pdf
  ✓ Loaded 8 pages

Total documents loaded: 13


In [5]:
# Text splitting get into chunks

def split_documents(documents, chunk_size =1000, chunk_overlap=200):
    """SPlit documents into smaler chunks for better RAG performance."""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size= chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators=["\n\n","\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [6]:

chunks=split_documents(all_pdf_documents)
chunks

Split 13 documents into 35 chunks

Example chunk:
Content: Arthritis Progression Prediction Using NLP 
 
 
 
Neelam More [20038484], nmore2@stevens.edu  
Viyasha Thakkar [20049574], vthakkar@stevens.edu 
 
Stevens Institution of Technology 
Fall 2025 
 
 
1 I...
Metadata: {'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2025-10-02T14:07:06-04:00', 'msip_label_a73fd474-4f3c-44ed-88fb-5cc4bd2471bf_enabled': 'true', 'msip_label_a73fd474-4f3c-44ed-88fb-5cc4bd2471bf_setdate': '2025-09-30T23:50:38Z', 'msip_label_a73fd474-4f3c-44ed-88fb-5cc4bd2471bf_method': 'Standard', 'msip_label_a73fd474-4f3c-44ed-88fb-5cc4bd2471bf_name': 'defa4170-0d19-0005-0004-bc88714345d2', 'msip_label_a73fd474-4f3c-44ed-88fb-5cc4bd2471bf_siteid': '8d1a69ec-03b5-4345-ae21-dad112f5fb4f', 'msip_label_a73fd474-4f3c-44ed-88fb-5cc4bd2471bf_actionid': 'b662e1f7-f3ca-41a6-b121-6b6adccac906', 'msip_label_a73fd474-4f3c-44ed-88fb-5cc4bd2471bf_contentbits': '0', 'm

[Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2025-10-02T14:07:06-04:00', 'msip_label_a73fd474-4f3c-44ed-88fb-5cc4bd2471bf_enabled': 'true', 'msip_label_a73fd474-4f3c-44ed-88fb-5cc4bd2471bf_setdate': '2025-09-30T23:50:38Z', 'msip_label_a73fd474-4f3c-44ed-88fb-5cc4bd2471bf_method': 'Standard', 'msip_label_a73fd474-4f3c-44ed-88fb-5cc4bd2471bf_name': 'defa4170-0d19-0005-0004-bc88714345d2', 'msip_label_a73fd474-4f3c-44ed-88fb-5cc4bd2471bf_siteid': '8d1a69ec-03b5-4345-ae21-dad112f5fb4f', 'msip_label_a73fd474-4f3c-44ed-88fb-5cc4bd2471bf_actionid': 'b662e1f7-f3ca-41a6-b121-6b6adccac906', 'msip_label_a73fd474-4f3c-44ed-88fb-5cc4bd2471bf_contentbits': '0', 'msip_label_a73fd474-4f3c-44ed-88fb-5cc4bd2471bf_tag': '10, 3, 0, 1', 'author': 'Neelam More', 'moddate': '2025-10-02T14:07:06-04:00', 'source': '../data/pdf/25Fall_CS584_NLP_Project_Proposal_Neelam_Viyasha (2).pdf', 'total_pages': 3, 'page': 0, 'page_labe

## embedding and VectorStoreDB

In [10]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity


/Users/neelammore/Documents/Study/AI/03_AI/03_RAG/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model =None
        self._load_model()


    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager



Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 20015.44it/s]


Model loaded successfully. Embedding dimension: 384


/var/folders/yr/lrly8x3n77333mlf8rs4qvy80000gn/T/ipykernel_2594/4289707575.py:21: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


## VectorStore

In [13]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore
    

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [14]:
chunks

[Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2025-10-02T14:07:06-04:00', 'msip_label_a73fd474-4f3c-44ed-88fb-5cc4bd2471bf_enabled': 'true', 'msip_label_a73fd474-4f3c-44ed-88fb-5cc4bd2471bf_setdate': '2025-09-30T23:50:38Z', 'msip_label_a73fd474-4f3c-44ed-88fb-5cc4bd2471bf_method': 'Standard', 'msip_label_a73fd474-4f3c-44ed-88fb-5cc4bd2471bf_name': 'defa4170-0d19-0005-0004-bc88714345d2', 'msip_label_a73fd474-4f3c-44ed-88fb-5cc4bd2471bf_siteid': '8d1a69ec-03b5-4345-ae21-dad112f5fb4f', 'msip_label_a73fd474-4f3c-44ed-88fb-5cc4bd2471bf_actionid': 'b662e1f7-f3ca-41a6-b121-6b6adccac906', 'msip_label_a73fd474-4f3c-44ed-88fb-5cc4bd2471bf_contentbits': '0', 'msip_label_a73fd474-4f3c-44ed-88fb-5cc4bd2471bf_tag': '10, 3, 0, 1', 'author': 'Neelam More', 'moddate': '2025-10-02T14:07:06-04:00', 'source': '../data/pdf/25Fall_CS584_NLP_Project_Proposal_Neelam_Viyasha (2).pdf', 'total_pages': 3, 'page': 0, 'page_labe

In [15]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 35 texts...


Batches: 100%|██████████| 2/2 [00:02<00:00,  1.01s/it]

Generated embeddings with shape: (35, 384)
Adding 35 documents to vector store...
Successfully added 35 documents to vector store
Total documents in collection: 35


### Retriever Pipeline From VectorStore

In [16]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [17]:
rag_retriever


In [18]:
rag_retriever.retrieve("What is attention is all you need")


Retrieving documents for query: 'What is attention is all you need'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.28it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)


[]

In [19]:
rag_retriever.retrieve("Unified Multi-task Learning Framework")


Retrieving documents for query: 'Unified Multi-task Learning Framework'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.14it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)


[]

### RAG Pipeline- VectorDB To LLM Output Generation

In [21]:

import os
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")


In [23]:

from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage

In [26]:
class OpenAILLM:
    def __init__(
        self,
        model_name: str = "gpt-4.1",
        api_key: str | None = None,
    ):
        """
        Initialize an OpenAI chat model.

        Args:
            model_name: OpenAI model name.
            api_key: OpenAI API key. If omitted, OPENAI_API_KEY is used.
        """
        self.model_name = model_name
        self.api_key = api_key or os.getenv("OPENAI_API_KEY")

        if not self.api_key:
            raise ValueError(
                "OpenAI API key is required. Set the OPENAI_API_KEY "
                "environment variable or pass api_key."
            )

        self.llm = ChatOpenAI(
            api_key=self.api_key,
            model=self.model_name,
            temperature=0.1,
            max_tokens=1024,
        )

        print(f"Initialized OpenAI LLM with model: {self.model_name}")

    def generate_response(
        self,
        query: str,
        context: str,
        max_length: int = 500,
    ) -> str:
        prompt_template = PromptTemplate(
            input_variables=["context", "question"],
            template="""You are a helpful AI assistant.

Use only the following context to answer the question accurately.

Context:
{context}

Question:
{question}

If the context does not contain enough information, say that the provided
context is insufficient.

Answer:""",
        )

        formatted_prompt = prompt_template.format(
            context=context,
            question=query,
        )

        try:
            response = self.llm.invoke(
                [HumanMessage(content=formatted_prompt)]
            )
            return str(response.content)

        except Exception as e:
            return f"Error generating response: {e}"

    def generate_response_simple(
        self,
        query: str,
        context: str,
    ) -> str:
        simple_prompt = f"""Based on the following context:

{context}

Question:
{query}

Answer:"""

        try:
            response = self.llm.invoke(
                [HumanMessage(content=simple_prompt)]
            )
            return str(response.content)

        except Exception as e:
            return f"Error generating response: {e}"


try:
    openai_llm = OpenAILLM(
        api_key=os.getenv("OPENAI_API_KEY")
    )
    print("OpenAI LLM initialized successfully!")

except ValueError as e:
    print(f"Warning: {e}")
    print("Please set your OPENAI_API_KEY environment variable.")
    openai_llm = None

Initialized OpenAI LLM with model: gpt-4.1
OpenAI LLM initialized successfully!


In [27]:

# Initialize Groq LLM (you'll need to set GROQ_API_KEY environment variable)
try:
    openai_llm = OpenAILLM(api_key=os.getenv("OPENAI_API_KEY"))
    print("OPENAAI LLM initialized successfully!")
except ValueError as e:
    print(f"Warning: {e}")
    print("Please set your OPENAI_API_KEY environment variable to use the LLM.")
    groq_llm = None


Initialized OpenAI LLM with model: gpt-4.1
OPENAAI LLM initialized successfully!


In [28]:

### get the context from the retriever and pass it to the LLM

rag_retriever.retrieve("Unified Multi-task Learning Framework")


Retrieving documents for query: 'Unified Multi-task Learning Framework'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  5.52it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)


[]

### Integration Vectordb Context pipeline With LLM output

In [29]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="gemma2-9b-it",temperature=0.1,max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [30]:
answer=rag_simple("What is attention mechanism?",rag_retriever,llm)
print(answer)


Retrieving documents for query: 'What is attention mechanism?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.76it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)
No relevant context found to answer the question.


### Enhanced RAG Pipeline Features

In [31]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("Hard Negative Mining Technqiues", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'Hard Negative Mining Technqiues'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 19.06it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)
Answer: No relevant context found.
Sources: []
Confidence: 0.0
Context Preview: 


In [32]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("what is attention is all you need", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'what is attention is all you need'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 22.52it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)


BadRequestError: Error code: 400 - {'error': {'message': 'The model `gemma2-9b-it` has been decommissioned and is no longer supported. Please refer to https://console.groq.com/docs/deprecations for a recommendation on which model to use instead.', 'type': 'invalid_request_error', 'code': 'model_decommissioned'}}